# 🗣️ Speech-AI-Forge Colab

👋本脚本基于 [Speech-AI-Forge](https://github.com/lenML/Speech-AI-Forge) 构建。如果此项目对你有帮助，欢迎到 github 为我们 star 支持！也欢迎提交 pr issues~

## 运行指南

1. 在菜单栏中选择 **代码执行程序**。
2. 点击 **全部运行**。

运行完成后，请在下方日志中找到如下信息：

```
Running on public URL: https://**.gradio.live
```

该链接即为您可以访问的公网地址。

> 注意：如果在安装包时提示需要重启，请选择 "否"。

In [ ]:
# @markdown ## 环境部署

# 1. Clone the repository
!git clone https://github.com/lenML/Speech-AI-Forge

# 2. Change directory to the repository
%cd Speech-AI-Forge

# 3. Install ffmpeg / rubberband-cli / sox
# !apt-get update -y
!apt-get install -y ffmpeg rubberband-cli sox

# 4. Install dependencies
# 编辑 requirements.txt 找到 `### PyTorch Dependencies` 删除之后的依赖，不用 colab 内置的pytorch会报错
!sed '/^### PyTorch Dependencies$/,$d' requirements.txt > requirements.colab.txt
%pip install uv
!uv pip install -r requirements.colab.txt --index-strategy unsafe-best-match --force-reinstall
# 这个比较特殊...在colab环境需要单独安装
%pip install protobuf==4.25.3

In [ ]:
import os

from modules.downloader.AutoModelDownloader import AutoModelDownloader

# @markdown ## 模型下载
# @markdown 大部分模型的大小接近 2GB，请确保有足够的存储空间和网络带宽。  <br/>
# @markdown > 这里只是选择预下载模型，开启webui后，使用未下载模型也会自动下载 <br/>
# @markdown > 至少必须选择一个 TTS 模型。如果没有选择，将默认下载 `Qwen3-TTS` 。

# @markdown ### Hugging Face Token (可选)
# @markdown 部分模型可能需要配置 Hugging Face Token 才能下载。 如果您需要使用这些模型,请在此处输入您的 Token。您可以从 [Hugging Face](https://huggingface.co/settings/tokens) 获取您的 Token.
HF_TOKEN = "" # @param {"type":"string","placeholder":"put huggingface token here..."}

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN  # 设置 "HF_TOKEN" 环境变量
    print("✍HF_TOKEN 环境变量已配置。")  # Helpful feedback
else:
    print("🤟未提供 HF_TOKEN, 跳过环境变量配置.")  # Good to know why

# @markdown ### TTS 模型

# @markdown - Qwen3-TTS: [GitHub](https://github.com/QwenLM/Qwen3-TTS) - Qwen3-TTS is an open-source series of TTS models developed by the Qwen team at Alibaba Cloud, supporting stable, expressive, and streaming speech generation, free-form voice design, and vivid voice cloning.
# @markdown - Index-TTS: [GitHub](https://github.com/index-tts/index-tts) - An Industrial-Level Controllable and Efficient Zero-Shot Text-To-Speech System
# @markdown - CosyVoice: [GitHub](https://github.com/FunAudioLLM/CosyVoice) - Multi-lingual large voice generation model, providing inference, training and deployment full-stack ability.
# @markdown - FishSpeech: [GitHub](https://github.com/fishaudio/fish-speech) - Brand new TTS solution
# @markdown - GPT-SoVITS: [GitHub](https://github.com/RVC-Boss/GPT-SoVITS) - 1 min voice data can also be used to train a good TTS model! (few shot voice cloning)
# @markdown - Spark-TTS: [GitHub](https://github.com/SparkAudio/Spark-TTS) - Spark-TTS Inference
# @markdown - ChatTTS: [GitHub](https://github.com/2noise/ChatTTS) - A generative speech model for daily dialogue.
# @markdown - F5-TTS: [GitHub](https://github.com/SWivid/F5-TTS) - A Fairytaler that Fakes Fluent and Faithful Speech with Flow Matching
# @markdown - FireRedTTS: [GitHub](https://github.com/FireRedTeam/FireRedTTS) - An Open-Sourced LLM-empowered Foundation TTS System
download_qwen3_tts = True  # @param {"type":"boolean"}

download_index_tts = False  # @param {"type":"boolean"}

download_cosyvoice = False  # @param {"type":"boolean"}

download_fish_speech = False  # @param {"type":"boolean"}

download_gpt_sovits_v4 = False  # @param {"type":"boolean"}

download_spark_tts = False  # @param {"type":"boolean"}

download_chattts = False  # @param {"type":"boolean"}

download_f5_tts = False  # @param {"type":"boolean"}

download_fire_red_tts = False  # @param {"type":"boolean"}

# @markdown ### ASR 模型
# @markdown - SenseVoice: [GitHub](https://github.com/FunAudioLLM/SenseVoice) - Multilingual Voice Understanding Model
# @markdown - Whisper: [GitHub](https://github.com/openai/whisper) - Robust Speech Recognition via Large-Scale Weak Supervision
download_sense_voice = False  # @param {"type":"boolean"}

download_whisper = False  # @param {"type":"boolean"}

# @markdown ### Clone Voice 模型
# @markdown OpenVoice: [GitHub](https://github.com/myshell-ai/OpenVoice) - Instant voice cloning by MIT and MyShell.
download_open_voice = False  # @param {"type":"boolean"}

# @markdown ### 人声增强模型
# @markdown resemble-enhance: [GitHub](https://github.com/resemble-ai/resemble-enhance) - AI powered speech denoising and enhancement
download_enhancer = False  # @param {"type":"boolean"}

# | 模型类别       | 内部模型 ID（可直接用于 `--models`） |
# |----------------|----------------------------------------|
# | **TTS**        | `ChatTTS`                              |
# |                | `CosyVoice2-0.5B`                      |
# |                | `CosyVoice_300M_Instruct`              |
# |                | `Fun-CosyVoice3-0.5B-2512`              |
# |                | `F5-TTS-V1`                            |
# |                | `FireRedTTS`                           |
# |                | `fish-speech-1_4`                      |
# |                | `fish-speech-1.2-sft`                  |
# |                | `Index-TTS-1.5`                        |
# |                | `Index-TTS`                            |
# |                | `Index-TTS-2`                          |
# |                | `Qwen3-TTS-12Hz-0.6B-Base`             |
# |                | `Qwen3-TTS-12Hz-0.6B-CustomVoice`      |
# |                | `Qwen3-TTS-12Hz-1.7B-Base`             |
# |                | `Qwen3-TTS-12Hz-1.7B-CustomVoice`      |
# |                | `Qwen3-TTS-12Hz-1.7B-VoiceDesign`      |
# |                | `Spark-TTS-0.5B`                       |
# |                | `gpt_sovits_v4`                        |
# | **CV / Voice Clone** | `OpenVoiceV2`                     |
# | **Enhancer**   | `resemble-enhance`                     |
# | **依赖模型（Index-TTS-2 所需）** | `amphion/MaskGCT`       |
# |                                 | `nvidia/bigvgan_v2_22khz_80band_256x` |
# |                                 | `funasr/campplus`                      |
# |                                 | `facebook/w2v-bert-2.0`               |
# |                                 | `vocos-mel-24khz`                      |

# 检查是否至少选择了一个 TTS 模型
if not any(
    [
        download_qwen3_tts,
        
        download_chattts,
        download_fish_speech,
        download_cosyvoice,
        download_fire_red_tts,
        download_f5_tts,
        download_index_tts,
        download_spark_tts,
    ]
):
    print("⚠️ 未选择任何 TTS 模型，默认下载 Qwen3-TTS...")
    download_qwen3_tts = True

dl_source = "huggingface"
# 因为支持自动下载，所以只预先下载一个最新模型，不然colab磁盘装不下
download_cfg = [
    ["ChatTTS", download_chattts],
    
    ["CosyVoice2-0.5B", download_cosyvoice],
    # ["CosyVoice_300M_Instruct", download_cosyvoice],
    
    ["Fun-CosyVoice3-0.5B-2512", download_cosyvoice],
    ["F5-TTS-V1", download_f5_tts],
    ["FireRedTTS", download_fire_red_tts],
    ["fish-speech-1_4", download_fish_speech],
    ["fish-speech-1.2-sft", download_fish_speech],
    
    
    ["Index-TTS-2", download_index_tts],
    # ["Index-TTS-1.5", download_index_tts],
    # ["Index-TTS", download_index_tts],
    
    ["Qwen3-TTS-12Hz-1.7B-Base", download_qwen3_tts],
    # ["Qwen3-TTS-12Hz-0.6B-Base", download_qwen3_tts],
    # ["Qwen3-TTS-12Hz-0.6B-CustomVoice", download_qwen3_tts],
    # ["Qwen3-TTS-12Hz-1.7B-CustomVoice", download_qwen3_tts],
    # ["Qwen3-TTS-12Hz-1.7B-VoiceDesign", download_qwen3_tts],
    
    ["Spark-TTS-0.5B", download_spark_tts],
    ["gpt_sovits_v4", download_gpt_sovits_v4],
    # index-tts 依赖
    ["amphion/MaskGCT", download_index_tts],
    ["nvidia/bigvgan_v2_22khz_80band_256x", download_index_tts],
    ["funasr/campplus", download_index_tts],
    ["facebook/w2v-bert-2.0", download_index_tts],
    ["vocos-mel-24khz", download_index_tts],
    # f5 tts 依赖
    ["vocos-mel-24khz", download_f5_tts],
    # enhancer
    ["resemble-enhance", download_enhancer],
    # open voice
    ["open-voice", download_open_voice],
    # asr
    ["faster-whisper-large-v3-turbo-ct2", download_whisper],
    ["SenseVoiceSmall", download_sense_voice],
    ["fsmn-vad", download_sense_voice],
]

# 下载模型
adl = AutoModelDownloader(down_source=dl_source)
ids = [ model_name for model_name, should_download in download_cfg if should_download ]
print(f"📦 即将下载模型：{', '.join(ids)}")
adl.download_models(ids,force=True)

print("✅ 所有选定模型已下载完成")

## 运行 WebUI

In [ ]:
!nvcc --version

In [ ]:
!nvidia-smi

In [ ]:
!python webui.py --share --language=zh-CN --off_track_tqdm